In [120]:
import numpy as np
import pandas as pd
import pandas_ta as ta
import gymnasium as gym
import gym_trading_env
from gym_trading_env.wrapper import DiscreteActionsWrapper
from stable_baselines3 import DQN, PPO

# pip install sb3-contrib

In [121]:
def reward_function(history):   
    current_val = history["portfolio_valuation", -1]
    last_val = history["portfolio_valuation", -2]
    reward = (current_val / last_val) - 1
    
    if current_val == 0:
        return -1.0
    
    return reward

In [122]:
def preprocess1(df):
    df = df.copy()
    df = df.sort_index()
    
    # Volume Security (Prevents division by zero)
    if "volume" not in df.columns or (df["volume"] == 0).all():
        df["volume"] = 1.0 
        has_volume = False
    else:
        df["volume"] = df["volume"].replace(0, 1e-5)
        has_volume = True


    # RSI (Momentum)
    df["feature_rsi"] = df.ta.rsi(length=14) / 100.0

    # ADX (Trend Strength)
    adx_df = df.ta.adx(length=14)
    if adx_df is not None and "ADX_14" in adx_df.columns:
        df["feature_adx"] = adx_df["ADX_14"] / 100.0
    else:
        df["feature_adx"] = 0

    # Bollinger Bands (Position & Volatility)
    bb = df.ta.bbands(length=20, std=2)
    bbp_col = bb.columns[bb.columns.str.startswith("BBP")][0]
    bbb_col = bb.columns[bb.columns.str.startswith("BBB")][0]
    df["feature_bb_pos"] = bb[bbp_col] 
    df["feature_bb_width"] = bb[bbb_col] / 100.0

    # Moving Averages (Context)
    df["ma_20"] = df.ta.sma(length=20)
    df["feature_dist_ma_20"] = (df["close"] / df["ma_20"]) - 1
    df["ma_50"] = df.ta.sma(length=50)
    df["feature_dist_ma_50"] = (df["close"] / df["ma_50"]) - 1
    df["ma_200"] = df.ta.sma(length=200)
    df["feature_dist_ma_200"] = (df["close"] / df["ma_200"]) - 1

    # Log Returns (Immediate price action)
    df["feature_log_ret"] = np.log(df["close"] / df["close"].shift(1)).fillna(0)

    # Volume Metrics
    if has_volume:
        vol_ma = df.ta.sma(close=df["volume"], length=20)
        df["feature_vol_rel"] = (df["volume"] / vol_ma) - 1
        
        # OBV Slope (Smart Money flow)
        df["obv"] = df.ta.obv()
        df["feature_obv_slope"] = df["obv"].pct_change(5).fillna(0)
    else:
        df["feature_vol_rel"] = 0
        df["feature_obv_slope"] = 0

    # Handle Infinite values
    df.replace([np.inf, -np.inf], 0, inplace=True)
    
    # Drop NaNs (Crucial for MA 200)
    df.dropna(inplace=True)

    # CLIPPING (Crucial for PPO Stability)
    # Forces all features to stay between -10 and 10 to prevent exploding gradients
    features_to_clip = [c for c in df.columns if "feature_" in c]
    df[features_to_clip] = df[features_to_clip].clip(-10, 10)

    return df

# def preprocess1(df):
#     df = df.copy()
#     df = df.sort_index()
    
#     # Price Position
#     # Bollinger band (price according to min and max)
#     sma = df["close"].rolling(window=20).mean()
#     std = df["close"].rolling(window=20).std()
#     upper = sma + (2 * std)
#     lower = sma - (2 * std)
#     df["feature_bb_pos"] = (df["close"] - lower) / (upper - lower) # Position (0 = low, 1 = high, >1 = higher than before)
#     df["feature_bb_width"] = (upper - lower) / df["close"]
    
#     # Trend
#     # Distance to Moving Average 20
#     df["ma_20"] = df["close"].rolling(window=20).mean()
#     df["feature_dist_ma_20"] = df["close"] / df["ma_20"] - 1
#     # Distance to Moving Average 50
#     df["ma_50"] = df["close"].rolling(window=50).mean()
#     df["feature_dist_ma_50"] = df["close"] / df["ma_50"] - 1
#     # Distance to Moving Average 200
#     df["ma_200"] = df["close"].rolling(window=200).mean()
#     df["feature_dist_ma_200"] = df["close"] / df["ma_200"] - 1
    
    
#     # Trend speed
#     df["feature_log_ret"] = np.log(df["close"] / df["close"].shift(1))
    
#     # Trend Strength
#     # ADX Simplified
#     high_low = df["high"] - df["low"]
#     high_close = np.abs(df["high"] - df["close"].shift())
#     low_close = np.abs(df["low"] - df["close"].shift())
#     ranges = pd.concat([high_low, high_close, low_close], axis=1)
#     true_range = np.max(ranges, axis=1)
#     atr = true_range.rolling(14).mean() + 1e-10
#     # Directional Movement
#     up_move = df["high"] - df["high"].shift()
#     down_move = df["low"].shift() - df["low"]
#     plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
#     minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
#     plus_di = 100 * (pd.Series(plus_dm, index=df.index).rolling(14).mean() / atr)
#     minus_di = 100 * (pd.Series(minus_dm, index=df.index).rolling(14).mean() / atr)
#     # DX et ADX
#     sum_di = plus_di + minus_di + 1e-10
#     dx = 100 * np.abs(plus_di - minus_di) / sum_di
#     df["feature_adx"] = dx.rolling(14).mean() / 100.0

#     # Volatility
#     df["feature_std_deviation"] = df["close"].rolling(window=20).std() / df["close"]
#     df["feature_candle_range"] = (df["high"] - df["low"]) / df["close"]

#     # Momentum
#     # RSI
#     delta = df["close"].diff()
#     gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
#     loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
#     rs = gain / loss
#     df["rsi"] = 100 - (100 / (1 + rs))
#     df["feature_rsi_norm"] = df["rsi"] / 100.0
    
#     # Volume
#     # Test if Volume == 0
#     if "volume" not in df.columns or (df["volume"] == 0).all():
#         df["volume"] = 1.0 # Value with no impact to prevent crash
#         has_volume = False
#     else:
#         df["volume"] = df["volume"].replace(0, 1e-5) # Replace 0 with low values
#         has_volume = True
#     # Calculations
#     if has_volume:
#         # Volume Relative Strength
#         df["vol_ma_20"] = df["volume"].rolling(window=20).mean()
#         df["feature_vol_rel"] = (df["volume"] / df["vol_ma_20"] - 1).fillna(0)
#         # OBV Slope
#         df["obv"] = (np.sign(df["close"].diff()) * df["volume"]).fillna(0).cumsum()
#         df["feature_obv_slope"] = df["obv"].pct_change(5).fillna(0)
#     else:
#         df["feature_vol_rel"] = 0.0
#         df["feature_obv_slope"] = 0.0
        
#     # Clean up
#     df.dropna(inplace=True) # Remove NaN
#     df.replace([np.inf, -np.inf], 0, inplace=True) # Remove infinite values due to calculations
    
#     return df


def preprocess2(df):
    df = df.copy()
    df = df.sort_index()
    
    # Test if volume == 0
    if "volume" not in df.columns or (df["volume"] == 0).all():
        df["volume"] = 1.0 
        has_volume = False
    else:
        df["volume"] = df["volume"].replace(0, 1e-5)
        has_volume = True
    
    # RSI (Momentum)
    df["feature_rsi"] = df.ta.rsi(length=14) / 100.0

    # ADX (Trend Strength)
    adx_df = df.ta.adx(length=14)
    if adx_df is not None and "ADX_14" in adx_df.columns:
        df["feature_adx"] = adx_df["ADX_14"] / 100.0
    else:
        df["feature_adx"] = 0

    # Bollinger Bands (Price Position & Volatility)
    bb = df.ta.bbands(length=20, std=2)
    bbp_col = [c for c in bb.columns if c.startswith("BBP")][0]
    bbb_col = [c for c in bb.columns if c.startswith("BBB")][0]
    df["feature_bb_pos"] = bb[bbp_col] 
    df["feature_bb_width"] = bb[bbb_col] / 100.0

    # Log Returns
    df["feature_log_ret"] = np.log(df["close"] / df["close"].shift(1)).fillna(0)

    # Distance to Moving Average (Context)
    df["ma_20"] = df["close"].rolling(window=20).mean()
    df["feature_dist_ma_20"] = df["close"] / df["ma_20"] - 1
    df["ma_50"] = df["close"].rolling(window=50).mean()
    df["feature_dist_ma_50"] = df["close"] / df["ma_50"] - 1
    df["ma_200"] = df["close"].rolling(window=200).mean()
    df["feature_dist_ma_200"] = df["close"] / df["ma_200"] - 1

    # Relative Volume (Volume anomaly detection)
    if has_volume:
        vol_ma = df.ta.sma(close=df["volume"], length=20)
        df["feature_vol_rel"] = (df["volume"] / vol_ma) - 1
    else:
        df["feature_vol_rel"] = 0
        
    features_to_clip = [c for c in df.columns if "feature_" in c]
    df[features_to_clip] = df[features_to_clip].clip(-10, 10)
        
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.dropna()

    return df

In [123]:
base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess2,
    portfolio_initial_value=1_000,
    trading_fees=0.1/100,
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_function,
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)
print("obs :", obs)
print("reward :", reward)
print("terminated :", terminated)
print("truncated :", truncated)
print("info :", info)

obs : [1.                nan        nan        nan 0.00399282        nan
        nan        nan        nan 2.         1.9920784 ]
reward : 0.006942351671612279
terminated : False
truncated : False
info : {'idx': 1, 'step': 1, 'date': np.datetime64('2023-11-21T14:00:00.000000000'), 'position_index': 3, 'position': 2, 'real_position': np.float64(1.9920783858267341), 'data_ma_50': nan, 'data_high': 2009.699951171875, 'data_date_close': Timestamp('2023-11-21 15:00:00'), 'data_open': 1999.699951171875, 'data_close': 2007.5999755859375, 'data_ma_200': nan, 'data_volume': 36358.0, 'data_low': 1999.300048828125, 'data_ma_20': nan, 'portfolio_valuation': np.float64(1006.9423516716123), 'portfolio_distribution_asset': np.float64(0.9991572618709149), 'portfolio_distribution_fiat': 0, 'portfolio_distribution_borrowed_asset': 0, 'portfolio_distribution_borrowed_fiat': np.float64(998.9574182217968), 'portfolio_distribution_interest_asset': 0.0, 'portfolio_distribution_interest_fiat': np.float64(0.00

In [124]:
model_dqn = DQN(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.0001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.3, # Explore during 30% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model_ppo = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95, # Factor for trade-off of bias vs variance for Generalized Advantage Estimator
    clip_range=0.2, # Clipping parameter, helps to keep the update stable
    ent_coef=0.01, # Entropy Coefficient (Exploration)
)

model = model_ppo

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [125]:
# Model Training on ./data/*.pkl files
model.learn(total_timesteps=500_000, log_interval=10)

ValueError: Expected parameter logits (Tensor of shape (1, 4)) of distribution Categorical(logits: torch.Size([1, 4])) to satisfy the constraint IndependentConstraint(Real(), 1), but found invalid values:
tensor([[nan, nan, nan, nan]])

In [ ]:
# Test
obs, info = env.reset()
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    
    obs, reward, terminated, truncated, info = env.step(int(action))
    done = terminated or truncated

# Logs for rendering
env.unwrapped.save_for_render(dir="render_logs")

Market Return : 22.46%   |   Portfolio Return : 24.19%   |   
